# Chapter 07: Isometry and Similarity in Euclidean Space

_Source span: Coxeter, Introduction to Geometry, Chapter 7, printed pages 96-106; PDF pages 114-124._ The scanned PDF pages were rendered privately for orientation only. This notebook does not reproduce source prose, exercises, figures, screenshots, page crops, or layouts.

## Chapter Goal

This chapter is the three-dimensional counterpart to the plane chapters on isometry and similarity. We represent a spatial motion by an affine rule `x |-> Mx + t` and inspect distance preservation, determinant sign, fixed sets, and the behavior of spheres. The visuals make the classification visible: direct/opposite sense, central inversion, products of reflections, twists, dilative rotations, and sphere-preserving transformations.


## Computational Translation Guide

| Book idea | Computational model | What to inspect |
| --- | --- | --- |
| Sense of a tetrahedron | Sign of `det([B-A, C-A, D-A])` | Direct maps keep the sign; opposite maps reverse it. |
| Spatial isometry | Affine map with orthogonal `M` | Pairwise distances stay fixed and `det(M)` is `+1` or `-1`. |
| Reflection in a plane | `x |-> x - 2(n dot x - d)n` | The mirror plane is fixed pointwise and the determinant is `-1`. |
| Central inversion | `x |-> 2O - x` | In 3D it is the product of three perpendicular plane reflections, hence opposite. |
| Product of two reflections | Intersecting planes rotate; parallel planes translate | The angle or translation length is twice the mirror separation. |
| Twist | Rotation about an axis plus translation along that axis | Orbits are helices with constant radius and constant axial step. |
| Dilative rotation | `x |-> O + lambda R_axis(theta)(x - O)` | Distances scale by `abs(lambda)` and orientation is the sign of `lambda^3`. |
| Sphere-preserving map | Similarity or sphere inversion composed with an isometry | Transformed sphere samples fit another sphere with tiny residual. |


## Compact Visual Storyboard Implemented

| Storyboard item | Artifact | Learner inspection target | Validation |
| --- | --- | --- | --- |
| Direct and opposite spatial isometries | `html/direct-opposite-spatial-frames.html` | Compare three tetrahedra and their moving frames. | Signed volume ratio equals `det(M)`; distances are preserved. |
| Central inversion and half-turn contrast | `figures/central-inversion-reflection-product.png` | See why `-I` reverses spatial sense while a half-turn preserves it. | Symbolic product of three coordinate-plane reflections equals `-I`. |
| Products of two reflections | `html/reflection-plane-products.html` | Intersecting mirrors rotate; parallel mirrors translate. | Measured result is twice the mirror angle or gap. |
| Twist traces | `html/twist-screw-traces.html` | Orbits wind around one axis without fixed points. | Radius, axial step, and pairwise distances are invariant. |
| Dilative rotation lab | `html/dilative-rotation-lab.html` and `tables/dilative_rotation_special_cases.csv` | Orbits spiral by simultaneous rotation and scaling. | Radial ratios equal `abs(lambda)`; `det(lambda R)=lambda**3`. |
| Sphere-preserving transformations | `html/sphere-preserving-transformations.html` | Similarity and sphere inversion both return spheres. | Fitted centers/radii match formulas. |


## Inspection Protocol

Read the chapter through invariants rather than through isolated pictures. A spatial isometry preserves all distances, but it can preserve or reverse orientation; the signed volume of a tetrahedron is the notebook's detector for that distinction. A reflection in one plane is opposite, while products of two reflections recover direct motions such as rotations or translations. Products of three reflections explain why central inversion behaves differently in space than a half-turn: the determinant records the difference even when both pictures look like point-centered motion.

The later artifacts separate isometries from similarities. A twist has no fixed point, but the orbit keeps its distance from the axis and advances by a constant step. A dilative rotation has a fixed center and combines rotation with a non-unit scale; the point trace makes the scale visible, and the determinant check explains why direct versus opposite depends on the sign of the cubic scale factor. The sphere-preservation lab closes the chapter by checking that inversion and similarity send sampled sphere points to another sphere with small residuals. In every case the visual asks where to look, while the JSON and CSV checks state what must remain true after the motion.


In [ ]:
from __future__ import annotations
import json, math, sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sympy as sp
from IPython.display import display

CHAPTER_NO = 7
HERE = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Run from inside the Introduction-to-Geometry course tree.")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from utils.artifacts import assert_artifacts, book_relative, display_artifact, ensure_chapter_artifact_dirs, write_csv, write_json

D = ensure_chapter_artifact_dirs(CHAPTER_NO, BOOK_ROOT)
PATHS = {
    "orientation_html": D["html"] / "direct-opposite-spatial-frames.html",
    "orientation_checks": D["checks"] / "orientation_determinant_checks.json",
    "orientation_table": D["tables"] / "isometry_orientation_table.csv",
    "central_png": D["figures"] / "central-inversion-reflection-product.png",
    "central_checks": D["checks"] / "central_inversion_checks.json",
    "reflection_html": D["html"] / "reflection-plane-products.html",
    "reflection_checks": D["checks"] / "reflection_product_checks.json",
    "twist_html": D["html"] / "twist-screw-traces.html",
    "twist_checks": D["checks"] / "twist_checks.json",
    "twist_data": D["data"] / "twist_orbit_samples.csv",
    "dilative_html": D["html"] / "dilative-rotation-lab.html",
    "dilative_checks": D["checks"] / "dilative_rotation_checks.json",
    "dilative_data": D["data"] / "dilative_rotation_orbits.csv",
    "special_cases": D["tables"] / "dilative_rotation_special_cases.csv",
    "sphere_html": D["html"] / "sphere-preserving-transformations.html",
    "sphere_checks": D["checks"] / "sphere_preservation_checks.json",
    "visual_summary": D["checks"] / "visual_summary.json",
    "final_sanity": D["checks"] / "final_sanity.json",
    "artifact_manifest": D["tables"] / "artifact_manifest.csv",
}
SOURCE_SPAN = {"printed_pages": "96-106", "pdf_pages": "114-124", "source_note": "Scanned PDF pages rendered privately for orientation only."}
COLORS = {"blue":"#2f6fbb","red":"#c43c39","green":"#3d8b5f","gold":"#d59f0f","purple":"#7357a6","gray":"#5d6875"}
np.set_printoptions(precision=6, suppress=True)

def unit(v):
    v=np.asarray(v,float); n=np.linalg.norm(v)
    if n==0: raise ValueError("zero vector")
    return v/n

def skew(axis):
    x,y,z=unit(axis); return np.array([[0,-z,y],[z,0,-x],[-y,x,0]],float)

def rotation_matrix(axis, theta):
    K=skew(axis); return np.eye(3)+math.sin(theta)*K+(1-math.cos(theta))*(K@K)

def reflection_matrix(normal):
    n=unit(normal); return np.eye(3)-2*np.outer(n,n)

def reflection_affine(normal, offset=0.0):
    n=unit(normal); return reflection_matrix(n), 2*offset*n

def compose_affine(M2,t2,M1,t1): return M2@M1, M2@t1+t2

def apply_affine(M,t,points): return np.asarray(points,float)@M.T+t

def signed_tetra_volume(points):
    A,B,C,D=np.asarray(points,float)
    return float(np.linalg.det(np.column_stack([B-A,C-A,D-A]))/6)

def pairwise_distances(points):
    p=np.asarray(points,float); return np.linalg.norm(p[:,None,:]-p[None,:,:],axis=-1)

def max_pairwise_error(a,b): return float(np.max(np.abs(pairwise_distances(a)-pairwise_distances(b))))

def write_plotly(fig,path,title):
    fig.update_layout(title=title,width=1020,height=640,margin=dict(l=0,r=0,b=0,t=58),legend=dict(orientation="h",yanchor="bottom",y=1.02,xanchor="left",x=0))
    fig.write_html(str(path), include_plotlyjs="cdn", full_html=True, include_mathjax=False)
    return path

def set_scene_layout(fig,names):
    for name in names:
        fig.update_layout(**{name:dict(aspectmode="data",xaxis_title="x",yaxis_title="y",zaxis_title="z",camera=dict(eye=dict(x=1.45,y=-1.65,z=1.15)))})

def tetra_edges_trace(points,name,color):
    edges=[(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]; xs=[]; ys=[]; zs=[]
    for i,j in edges:
        xs += [points[i,0],points[j,0],None]; ys += [points[i,1],points[j,1],None]; zs += [points[i,2],points[j,2],None]
    return go.Scatter3d(x=xs,y=ys,z=zs,mode="lines",name=name,line=dict(color=color,width=6))

def tetra_mesh_trace(points,name,color):
    return go.Mesh3d(x=points[:,0],y=points[:,1],z=points[:,2],i=[0,0,0,1],j=[1,2,3,2],k=[2,3,1,3],name=name,color=color,opacity=.22,showscale=False)

def add_frame(fig,M,t,row,col,prefix,scale=.62):
    for vec,label,color in zip(np.eye(3),["e1","e2","e3"],[COLORS["red"],COLORS["green"],COLORS["blue"]]):
        o=np.asarray(t,float); e=o+M@(scale*vec)
        fig.add_trace(go.Scatter3d(x=[o[0],e[0]],y=[o[1],e[1]],z=[o[2],e[2]],mode="lines+markers+text",text=["",label],textposition="top center",marker=dict(size=[2,4],color=color),line=dict(color=color,width=5),name=f"{prefix} {label}",showlegend=False),row=row,col=col)

def plane_surface(normal, offset=0, span=1.6, steps=2):
    n=unit(normal); trial=np.array([1.,0,0])
    if abs(np.dot(trial,n))>.9: trial=np.array([0.,1,0])
    u=unit(np.cross(n,trial)); v=unit(np.cross(n,u)); g=np.linspace(-span,span,steps); s,t=np.meshgrid(g,g)
    pts=offset*n+s[:,:,None]*u+t[:,:,None]*v
    return pts[:,:,0],pts[:,:,1],pts[:,:,2]

def add_plane(fig,normal,offset,row,col,color,name,span=1.5,opacity=.22):
    x,y,z=plane_surface(normal,offset,span)
    fig.add_trace(go.Surface(x=x,y=y,z=z,name=name,opacity=opacity,colorscale=[[0,color],[1,color]],showscale=False),row=row,col=col)

def sphere_points(center,radius,n_theta=32,n_phi=16):
    th=np.linspace(0,2*np.pi,n_theta,endpoint=False); ph=np.linspace(.12,np.pi-.12,n_phi); tt,pp=np.meshgrid(th,ph)
    return np.column_stack([center[0]+radius*np.sin(pp).ravel()*np.cos(tt).ravel(),center[1]+radius*np.sin(pp).ravel()*np.sin(tt).ravel(),center[2]+radius*np.cos(pp).ravel()])

def sphere_surface(center,radius,n_theta=36,n_phi=18):
    th=np.linspace(0,2*np.pi,n_theta); ph=np.linspace(0,np.pi,n_phi); tt,pp=np.meshgrid(th,ph)
    return center[0]+radius*np.sin(pp)*np.cos(tt), center[1]+radius*np.sin(pp)*np.sin(tt), center[2]+radius*np.cos(pp)

def add_sphere_surface(fig,center,radius,row,col,color,name,opacity=.18):
    x,y,z=sphere_surface(np.asarray(center,float),float(radius))
    fig.add_trace(go.Surface(x=x,y=y,z=z,colorscale=[[0,color],[1,color]],opacity=opacity,showscale=False,name=name),row=row,col=col)

def fit_sphere(points):
    p=np.asarray(points,float); A=np.column_stack([2*p,np.ones(len(p))]); b=np.sum(p*p,axis=1); c,*_=np.linalg.lstsq(A,b,rcond=None); center=c[:3]; radius=math.sqrt(max(0,c[3]+center@center)); return center,radius,np.linalg.norm(p-center,axis=1)-radius

def axis_equal_3d(ax,points):
    p=np.asarray(points,float); c=p.mean(axis=0); r=max(np.ptp(p[:,0]),np.ptp(p[:,1]),np.ptp(p[:,2]),2)/2
    ax.set_xlim(c[0]-r,c[0]+r); ax.set_ylim(c[1]-r,c[1]+r); ax.set_zlim(c[2]-r,c[2]+r)


## 1. Direct and Opposite Spatial Isometries

In the plane, direct and opposite isometries are detected by signed area. In space the analogous test is signed tetrahedron volume. Distances alone cannot distinguish the two cases, so this first lab records both the distance error and the determinant sign.


In [ ]:
base_tetra=np.array([[0,0,0],[1.25,.10,.05],[.22,1.05,.12],[.28,.25,1.10]],float)
base_volume=signed_tetra_volume(base_tetra)
rot=rotation_matrix([1,1,.55],math.radians(58)); mirror=reflection_matrix([1,-.15,.25])
examples=[
    {"case":"original frame","M":np.eye(3),"t":np.array([0,0,0.]),"color":COLORS["gray"]},
    {"case":"direct isometry","M":rot,"t":np.array([1.65,.18,.12]),"color":COLORS["blue"]},
    {"case":"opposite isometry","M":rot@mirror,"t":np.array([-1.45,.05,.25]),"color":COLORS["red"]},
]
fig=make_subplots(rows=1,cols=3,specs=[[{"type":"scene"},{"type":"scene"},{"type":"scene"}]],subplot_titles=[e["case"] for e in examples])
rows=[]
for col,item in enumerate(examples,1):
    image=apply_affine(item["M"],item["t"],base_tetra); det=float(np.linalg.det(item["M"])); vol=signed_tetra_volume(image)
    fig.add_trace(tetra_mesh_trace(image,item["case"],item["color"]),row=1,col=col); fig.add_trace(tetra_edges_trace(image,f"{item['case']} edges",item["color"]),row=1,col=col)
    fig.add_trace(go.Scatter3d(x=image[:,0],y=image[:,1],z=image[:,2],mode="markers+text",text=["A","B","C","D"],textposition="top center",marker=dict(size=5,color=item["color"]),name=f"{item['case']} vertices",showlegend=False),row=1,col=col)
    add_frame(fig,item["M"],item["t"],1,col,item["case"])
    rows.append({"case":item["case"],"det_linear_part":det,"signed_volume":vol,"volume_ratio_to_original":vol/base_volume,"orientation_class":"direct" if det>0 else "opposite","max_pairwise_distance_error":max_pairwise_error(base_tetra,image)})
set_scene_layout(fig,["scene","scene2","scene3"]); write_plotly(fig,PATHS["orientation_html"],"Direct and opposite spatial isometries: distance agrees, volume sense may flip")
orientation_table=pd.DataFrame(rows); write_csv(PATHS["orientation_table"],orientation_table.to_dict("records"))
orientation_checks={"chapter":CHAPTER_NO,"source_span":SOURCE_SPAN,"base_signed_volume":base_volume,"rows":rows,"max_distance_error":float(orientation_table["max_pairwise_distance_error"].max()),"direct_det_positive":bool(rows[1]["det_linear_part"]>0),"opposite_det_negative":bool(rows[2]["det_linear_part"]<0)}
write_json(PATHS["orientation_checks"],orientation_checks)
assert orientation_checks["max_distance_error"]<1e-12
assert abs(rows[1]["volume_ratio_to_original"]-rows[1]["det_linear_part"])<1e-12
assert abs(rows[2]["volume_ratio_to_original"]-rows[2]["det_linear_part"])<1e-12
display_artifact(PATHS["orientation_html"],width=980)
orientation_table


## 2. Central Inversion: A Point Reflection Is Opposite in Space

Central inversion sends every point through a center to the opposite point on the same line. With center at the origin it is `x |-> -x`. The dimensional subtlety is that a plane half-turn is direct, while the spatial central inversion has determinant `-1` and reverses sense.


In [ ]:
Hx=sp.diag(-1,1,1); Hy=sp.diag(1,-1,1); Hz=sp.diag(1,1,-1)
central_product=sp.simplify(Hz*Hy*Hx); half_turn_z=sp.diag(-1,-1,1)
assert central_product == -sp.eye(3); assert int(central_product.det())==-1; assert int(half_turn_z.det())==1
P=np.array([1.10,.75,.55]); Pinv=-P; Phalf=np.array([-P[0],-P[1],P[2]])
g=np.linspace(-1.3,1.3,2); xx,yy=np.meshgrid(g,g); zz=np.zeros_like(xx)
fig=plt.figure(figsize=(11.5,5.4)); axes=[fig.add_subplot(1,2,1,projection="3d"),fig.add_subplot(1,2,2,projection="3d")]
for ax in axes:
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z"); ax.view_init(elev=22,azim=-48); ax.grid(False)
ax=axes[0]; ax.set_title("central inversion = three coordinate-plane reflections")
ax.plot_surface(zz,xx,yy,alpha=.14,color="#e57373",linewidth=0); ax.plot_surface(xx,zz,yy,alpha=.14,color="#81c784",linewidth=0); ax.plot_surface(xx,yy,zz,alpha=.14,color="#64b5f6",linewidth=0)
ax.scatter([0],[0],[0],color="black",s=35); ax.scatter([P[0]],[P[1]],[P[2]],color=COLORS["blue"],s=50); ax.scatter([Pinv[0]],[Pinv[1]],[Pinv[2]],color=COLORS["red"],s=50)
ax.plot([P[0],Pinv[0]],[P[1],Pinv[1]],[P[2],Pinv[2]],color="black",linestyle="--",linewidth=1.2); ax.text(P[0],P[1],P[2],"  P"); ax.text(Pinv[0],Pinv[1],Pinv[2],"  -P"); ax.text(0,0,0,"  O"); axis_equal_3d(ax,np.vstack([P,Pinv,np.zeros(3)]))
ax=axes[1]; ax.set_title("half-turn about an axis is direct")
axis_line=np.array([[0,0,-1.25],[0,0,1.25]]); ax.plot(axis_line[:,0],axis_line[:,1],axis_line[:,2],color="black",linewidth=2.8)
ax.scatter([P[0]],[P[1]],[P[2]],color=COLORS["blue"],s=50); ax.scatter([Phalf[0]],[Phalf[1]],[Phalf[2]],color=COLORS["green"],s=50)
ax.plot([P[0],Phalf[0]],[P[1],Phalf[1]],[P[2],Phalf[2]],color=COLORS["green"],linestyle="--",linewidth=1.2); ax.text(P[0],P[1],P[2],"  P"); ax.text(Phalf[0],Phalf[1],Phalf[2],"  R_pi(P)"); axis_equal_3d(ax,np.vstack([P,Phalf,axis_line]))
fig.tight_layout(); fig.savefig(PATHS["central_png"],dpi=170,bbox_inches="tight"); plt.close(fig)
central_checks={"chapter":CHAPTER_NO,"source_span":SOURCE_SPAN,"coordinate_plane_reflection_product":str(central_product),"central_inversion_determinant":int(central_product.det()),"half_turn_determinant":int(half_turn_z.det()),"central_inversion_distance_error":float(abs(np.linalg.norm(P)-np.linalg.norm(Pinv))),"half_turn_distance_error":float(abs(np.linalg.norm(P)-np.linalg.norm(Phalf)))}
write_json(PATHS["central_checks"],central_checks)
assert central_checks["central_inversion_determinant"]==-1 and central_checks["half_turn_determinant"]==1
assert central_checks["central_inversion_distance_error"]<1e-12
display_artifact(PATHS["central_png"],width=880)
central_checks


## 3. Products of Two Reflections: Rotation or Translation

The product of two plane reflections is already enough to build two basic direct isometries. If the mirrors intersect, their line of intersection is the rotation axis and the rotation angle is twice the dihedral angle. If the mirrors are parallel, the product is a translation perpendicular to the mirrors and its length is twice the distance between them.


In [ ]:
def vertical_plane_normal(phi):
    return np.array([-math.sin(phi), math.cos(phi), 0.0])

alpha=math.radians(32); phi1,phi2=-alpha/2,alpha/2
M1,t1=reflection_affine(vertical_plane_normal(phi1),0); M2,t2=reflection_affine(vertical_plane_normal(phi2),0)
Mrot,trot=compose_affine(M2,t2,M1,t1); measured_angle=math.atan2(Mrot[1,0],Mrot[0,0])
parallel_gap=.70
Ma,ta=reflection_affine([1,0,0],-parallel_gap/2); Mb,tb=reflection_affine([1,0,0],parallel_gap/2)
Mtrans,ttrans=compose_affine(Mb,tb,Ma,ta)
p=np.array([1.05,.25,.35]); rot_path=np.vstack([p,apply_affine(M1,t1,[p])[0],apply_affine(Mrot,trot,[p])[0]])
q=np.array([-.15,-.55,.40]); trans_path=np.vstack([q,apply_affine(Ma,ta,[q])[0],apply_affine(Mtrans,ttrans,[q])[0]])
fig=make_subplots(rows=1,cols=2,specs=[[{"type":"scene"},{"type":"scene"}]],subplot_titles=["intersecting mirrors: rotation","parallel mirrors: translation"])
add_plane(fig,vertical_plane_normal(phi1),0,1,1,"#90caf9","mirror 1",span=1.35); add_plane(fig,vertical_plane_normal(phi2),0,1,1,"#ffcc80","mirror 2",span=1.35)
fig.add_trace(go.Scatter3d(x=[0,0],y=[0,0],z=[-1.25,1.25],mode="lines",line=dict(color="black",width=6),name="rotation axis"),row=1,col=1)
fig.add_trace(go.Scatter3d(x=rot_path[:,0],y=rot_path[:,1],z=rot_path[:,2],mode="lines+markers+text",text=["P","H1(P)","H2H1(P)"],textposition="top center",line=dict(color=COLORS["purple"],width=5),marker=dict(size=5),name="rotated path"),row=1,col=1)
add_plane(fig,[1,0,0],-parallel_gap/2,1,2,"#90caf9","parallel mirror 1",span=1.25); add_plane(fig,[1,0,0],parallel_gap/2,1,2,"#ffcc80","parallel mirror 2",span=1.25)
fig.add_trace(go.Scatter3d(x=trans_path[:,0],y=trans_path[:,1],z=trans_path[:,2],mode="lines+markers+text",text=["Q","H1(Q)","H2H1(Q)"],textposition="top center",line=dict(color=COLORS["green"],width=5),marker=dict(size=5),name="translated path"),row=1,col=2)
fig.add_trace(go.Scatter3d(x=[q[0],q[0]+ttrans[0]],y=[q[1],q[1]+ttrans[1]],z=[q[2],q[2]+ttrans[2]],mode="lines",line=dict(color="black",width=4,dash="dash"),name="net translation"),row=1,col=2)
set_scene_layout(fig,["scene","scene2"]); write_plotly(fig,PATHS["reflection_html"],"Two plane reflections: rotation when mirrors meet, translation when mirrors are parallel")
reflection_checks={"chapter":CHAPTER_NO,"source_span":SOURCE_SPAN,"intersecting_plane_angle_degrees":math.degrees(alpha),"measured_rotation_degrees":math.degrees(measured_angle),"expected_rotation_degrees":math.degrees(2*alpha),"rotation_angle_error":float(abs(measured_angle-2*alpha)),"parallel_plane_gap":parallel_gap,"measured_translation_vector":ttrans.tolist(),"expected_translation_vector":[2*parallel_gap,0,0],"translation_error":float(np.linalg.norm(ttrans-np.array([2*parallel_gap,0,0]))),"rotation_determinant":float(np.linalg.det(Mrot)),"translation_linear_determinant":float(np.linalg.det(Mtrans))}
write_json(PATHS["reflection_checks"],reflection_checks)
assert reflection_checks["rotation_angle_error"]<1e-12 and reflection_checks["translation_error"]<1e-12
assert abs(reflection_checks["rotation_determinant"]-1)<1e-12
display_artifact(PATHS["reflection_html"],width=980)
reflection_checks


## 4. Three Reflections and the Twist

Three reflections create the opposite side of the classification: reflection, glide reflection, and rotatory reflection or rotatory inversion, depending on the fixed set. The remaining direct isometry with no fixed point is the twist, also called a screw displacement. It is easiest to recognize from its orbits: every point keeps the same distance from the axis while advancing a fixed amount along the axis after each turn.


In [ ]:
twist_theta=math.radians(54); twist_step=.34
R_twist=rotation_matrix([0,0,1],twist_theta); t_twist=np.array([0,0,twist_step])
seeds=[np.array([1.05,0,-.55]),np.array([.55,.72,-.20]),np.array([1.35,-.42,.20])]
steps=11; records=[]; sample_paths=[]
fig=go.Figure(); fig.add_trace(go.Scatter3d(x=[0,0],y=[0,0],z=[-.85,3.15],mode="lines",line=dict(color="black",width=7),name="twist axis"))
for i,seed in enumerate(seeds):
    pts=[seed]
    for _ in range(1,steps): pts.append(R_twist@pts[-1]+t_twist)
    pts=np.array(pts); sample_paths.append(pts); radii=np.linalg.norm(pts[:,:2],axis=1)
    for k,point in enumerate(pts): records.append({"seed":i,"step":k,"x":point[0],"y":point[1],"z":point[2],"radius_to_axis":radii[k]})
    fig.add_trace(go.Scatter3d(x=pts[:,0],y=pts[:,1],z=pts[:,2],mode="lines+markers",marker=dict(size=4),line=dict(width=5),name=f"orbit {i+1}"))
orbit_df=pd.DataFrame(records); orbit_df.to_csv(PATHS["twist_data"],index=False)
radius_drift=orbit_df.groupby("seed")["radius_to_axis"].agg(lambda s: float(s.max()-s.min())).max()
z_step_errors=[]
for i in range(len(seeds)):
    zs=orbit_df.loc[orbit_df["seed"]==i,"z"].to_numpy(); z_step_errors.extend(np.diff(zs)-twist_step)
A=np.eye(3)-R_twist; fixed_candidate,*_=np.linalg.lstsq(A,t_twist,rcond=None); fixed_residual=float(np.linalg.norm(A@fixed_candidate-t_twist))
initial_distances=pairwise_distances(np.array([path[0] for path in sample_paths])); pair_drift=[]
for k in range(steps): pair_drift.append(float(np.max(np.abs(pairwise_distances(np.array([path[k] for path in sample_paths]))-initial_distances))))
fig.update_layout(scene=dict(aspectmode="data",xaxis_title="x",yaxis_title="y",zaxis_title="z",camera=dict(eye=dict(x=1.55,y=-1.55,z=1.2))))
write_plotly(fig,PATHS["twist_html"],"Twist traces: rotation about an axis plus translation along the same axis")
twist_checks={"chapter":CHAPTER_NO,"source_span":SOURCE_SPAN,"rotation_angle_degrees":math.degrees(twist_theta),"axial_step":twist_step,"linear_determinant":float(np.linalg.det(R_twist)),"max_radius_drift":float(radius_drift),"max_axial_step_error":float(np.max(np.abs(z_step_errors))),"fixed_point_least_squares_residual":fixed_residual,"max_pair_distance_drift":float(max(pair_drift))}
write_json(PATHS["twist_checks"],twist_checks)
assert abs(twist_checks["linear_determinant"]-1)<1e-12
assert twist_checks["max_radius_drift"]<1e-12 and twist_checks["max_axial_step_error"]<1e-12
assert twist_checks["fixed_point_least_squares_residual"]>.1 and twist_checks["max_pair_distance_drift"]<1e-12
display_artifact(PATHS["twist_html"],width=980)
twist_checks


## 5. Dilative Rotation: Similarity With a Center and an Axis

A non-isometric similarity in space has a fixed point. Once the fixed point is moved to the origin and the axis is chosen, the essential rule is `x |-> lambda R_axis(theta) x`. The sign of `lambda` now matters. In three dimensions `det(lambda R) = lambda^3`, so a negative dilation reverses orientation. The same formula contains rotations, half-turns, central inversion, reflections in a plane, ordinary dilations, and genuine dilative rotations as special cases.


In [ ]:
def dilative_orbit(center,lam,theta,seed,n_steps):
    R=rotation_matrix([0,0,1],theta); M=lam*R; t=center-M@center; pts=[np.asarray(seed,float)]
    for _ in range(1,n_steps): pts.append(M@pts[-1]+t)
    return np.array(pts),M,t
center=np.array([.12,-.18,.05])
dilative_cases=[
    {"name":"direct: lambda = 1.16","lambda":1.16,"theta":math.radians(36),"seed":center+np.array([.80,.05,.38]),"color":COLORS["blue"]},
    {"name":"opposite: lambda = -0.84","lambda":-.84,"theta":math.radians(36),"seed":center+np.array([1.10,-.16,.55]),"color":COLORS["red"]},
]
fig=make_subplots(rows=1,cols=2,specs=[[{"type":"scene"},{"type":"scene"}]],subplot_titles=[c["name"] for c in dilative_cases])
dilative_records=[]; dilative_rows=[]
for col,case in enumerate(dilative_cases,1):
    pts,M,t=dilative_orbit(center,case["lambda"],case["theta"],case["seed"],9); rel=pts-center; radii=np.linalg.norm(rel[:,:2],axis=1); ratios=radii[1:]/radii[:-1]; fixed=np.linalg.solve(np.eye(3)-M,t)
    fig.add_trace(go.Scatter3d(x=pts[:,0],y=pts[:,1],z=pts[:,2],mode="lines+markers+text",text=[str(i) for i in range(len(pts))],textposition="top center",line=dict(color=case["color"],width=5),marker=dict(size=5,color=case["color"]),name=case["name"]),row=1,col=col)
    fig.add_trace(go.Scatter3d(x=[center[0],center[0]],y=[center[1],center[1]],z=[center[2]-1.15,center[2]+1.15],mode="lines",line=dict(color="black",width=6),name="axis through center",showlegend=(col==1)),row=1,col=col)
    fig.add_trace(go.Scatter3d(x=[center[0]],y=[center[1]],z=[center[2]],mode="markers+text",text=["O"],textposition="top center",marker=dict(size=6,color="black"),name="fixed center",showlegend=(col==1)),row=1,col=col)
    for k,point in enumerate(pts): dilative_records.append({"case":case["name"],"step":k,"x":point[0],"y":point[1],"z":point[2],"radius_to_axis":radii[k]})
    dilative_rows.append({"case":case["name"],"lambda":case["lambda"],"theta_degrees":math.degrees(case["theta"]),"determinant":float(np.linalg.det(M)),"orientation":"direct" if np.linalg.det(M)>0 else "opposite","max_radial_ratio_error":float(np.max(np.abs(ratios-abs(case["lambda"])))),"fixed_center_error":float(np.linalg.norm(fixed-center))})
pd.DataFrame(dilative_records).to_csv(PATHS["dilative_data"],index=False)
set_scene_layout(fig,["scene","scene2"]); write_plotly(fig,PATHS["dilative_html"],"Dilative rotation lab: simultaneous rotation and scaling about one axis")
lam,a=sp.symbols("lambda angle", real=True); Rz=sp.Matrix([[sp.cos(a),-sp.sin(a),0],[sp.sin(a),sp.cos(a),0],[0,0,1]]); symbolic_det=sp.trigsimp(sp.factor((lam*Rz).det()))
assert symbolic_det==lam**3
special_rows=[
    {"case":"identity","theta":"0","lambda":"1","orientation":"direct","fixed_set":"all space"},
    {"case":"half-turn","theta":"pi","lambda":"1","orientation":"direct","fixed_set":"axis"},
    {"case":"rotation","theta":"nonzero","lambda":"1","orientation":"direct","fixed_set":"axis"},
    {"case":"plane reflection","theta":"pi","lambda":"-1","orientation":"opposite","fixed_set":"plane perpendicular to axis"},
    {"case":"central inversion","theta":"0","lambda":"-1","orientation":"opposite","fixed_set":"center"},
    {"case":"rotatory inversion","theta":"not 0 or pi","lambda":"-1","orientation":"opposite","fixed_set":"center"},
    {"case":"ordinary dilation","theta":"0","lambda":"not +/-1","orientation":"sign(lambda^3)","fixed_set":"center"},
    {"case":"genuine dilative rotation","theta":"nonzero","lambda":"not +/-1","orientation":"sign(lambda^3)","fixed_set":"center"},
]
write_csv(PATHS["special_cases"],special_rows)
dilative_checks={"chapter":CHAPTER_NO,"source_span":SOURCE_SPAN,"symbolic_determinant":str(symbolic_det),"rows":dilative_rows,"max_radial_ratio_error":float(max(r["max_radial_ratio_error"] for r in dilative_rows)),"max_fixed_center_error":float(max(r["fixed_center_error"] for r in dilative_rows))}
write_json(PATHS["dilative_checks"],dilative_checks)
assert dilative_checks["symbolic_determinant"]=="lambda**3" and dilative_checks["max_radial_ratio_error"]<1e-12 and dilative_checks["max_fixed_center_error"]<1e-12
display_artifact(PATHS["dilative_html"],width=980)
display(pd.DataFrame(special_rows))
pd.DataFrame(dilative_rows)


## 6. Sphere-Preserving Transformations

The last section extends circle-preserving transformations to inversive three-space. A similarity obviously sends a sphere to another sphere. Inversion in a sphere is less obvious from the formula, so the lab samples points on one sphere, applies unit sphere inversion `x |-> x / ||x||^2`, and fits a sphere to the result. The fitted center and radius are compared with the closed-form inversion formula.


In [ ]:
source_center=np.array([1.72,.55,.42]); source_radius=.58
samples=sphere_points(source_center,source_radius,n_theta=34,n_phi=17)
inverted=samples/np.sum(samples*samples,axis=1)[:,None]
fit_center,fit_radius,inversion_residuals=fit_sphere(inverted)
den=float(source_center@source_center-source_radius**2); formula_center=source_center/den; formula_radius=source_radius/abs(den)
sim_center=np.array([-.35,-.20,.15]); sim_lambda=1.28; sim_rotation=rotation_matrix([.6,.2,1.0],math.radians(42)); sim_M=sim_lambda*sim_rotation; sim_t=sim_center-sim_M@sim_center
similarity_image=apply_affine(sim_M,sim_t,samples); sim_fit_center,sim_fit_radius,sim_residuals=fit_sphere(similarity_image)
expected_sim_center=sim_M@source_center+sim_t; expected_sim_radius=abs(sim_lambda)*source_radius
fig=make_subplots(rows=1,cols=3,specs=[[{"type":"scene"},{"type":"scene"},{"type":"scene"}]],subplot_titles=["source sphere","after unit sphere inversion","after a similarity"])
fig.add_trace(go.Scatter3d(x=samples[:,0],y=samples[:,1],z=samples[:,2],mode="markers",marker=dict(size=2.6,color=COLORS["blue"]),name="source samples"),row=1,col=1); add_sphere_surface(fig,source_center,source_radius,1,1,"#90caf9","source fitted sphere")
fig.add_trace(go.Scatter3d(x=inverted[:,0],y=inverted[:,1],z=inverted[:,2],mode="markers",marker=dict(size=2.6,color=COLORS["red"]),name="inverted samples"),row=1,col=2); add_sphere_surface(fig,fit_center,fit_radius,1,2,"#ef9a9a","inverted fitted sphere")
fig.add_trace(go.Scatter3d(x=similarity_image[:,0],y=similarity_image[:,1],z=similarity_image[:,2],mode="markers",marker=dict(size=2.6,color=COLORS["green"]),name="similarity samples"),row=1,col=3); add_sphere_surface(fig,sim_fit_center,sim_fit_radius,1,3,"#a5d6a7","similarity fitted sphere")
set_scene_layout(fig,["scene","scene2","scene3"]); write_plotly(fig,PATHS["sphere_html"],"Sphere-preserving transformations: similarity and inversion both return spheres")
sphere_checks={"chapter":CHAPTER_NO,"source_span":SOURCE_SPAN,"unit_inversion_formula_center":formula_center.tolist(),"unit_inversion_fit_center":fit_center.tolist(),"unit_inversion_formula_radius":formula_radius,"unit_inversion_fit_radius":fit_radius,"unit_inversion_center_error":float(np.linalg.norm(fit_center-formula_center)),"unit_inversion_radius_error":float(abs(fit_radius-formula_radius)),"unit_inversion_max_fit_residual":float(np.max(np.abs(inversion_residuals))),"similarity_expected_center":expected_sim_center.tolist(),"similarity_fit_center":sim_fit_center.tolist(),"similarity_expected_radius":expected_sim_radius,"similarity_fit_radius":sim_fit_radius,"similarity_center_error":float(np.linalg.norm(sim_fit_center-expected_sim_center)),"similarity_radius_error":float(abs(sim_fit_radius-expected_sim_radius)),"similarity_max_fit_residual":float(np.max(np.abs(sim_residuals)))}
write_json(PATHS["sphere_checks"],sphere_checks)
assert sphere_checks["unit_inversion_center_error"]<1e-12 and sphere_checks["unit_inversion_radius_error"]<1e-12 and sphere_checks["unit_inversion_max_fit_residual"]<1e-12
assert sphere_checks["similarity_center_error"]<1e-12 and sphere_checks["similarity_radius_error"]<1e-12 and sphere_checks["similarity_max_fit_residual"]<1e-12
display_artifact(PATHS["sphere_html"],width=980)
sphere_checks


## Applied Lab: Classify an Affine Motion From Its Invariants

The small classifier below is not meant to replace the synthetic proofs. Its job is to give a quick diagnostic for examples. Feed it an affine rule `x |-> Mx + t`; it reports whether the rule preserves distances, whether it is direct or opposite, and what fixed-point residual remains. The examples match the chapter's classification: rotations and translations are direct, central inversion and reflections are opposite, and a twist is direct but has no fixed point.


In [ ]:
def classify_affine(M,t,distance_tol=1e-10):
    M=np.asarray(M,float); t=np.asarray(t,float); gram_error=float(np.linalg.norm(M.T@M-np.eye(3))); det=float(np.linalg.det(M)); A=np.eye(3)-M
    fixed_candidate,*_=np.linalg.lstsq(A,t,rcond=None); fixed_residual=float(np.linalg.norm(A@fixed_candidate-t))
    return {"isometry":gram_error<distance_tol,"determinant":det,"orientation":"direct" if det>0 else "opposite","orthogonality_error":gram_error,"has_fixed_point":fixed_residual<1e-9,"fixed_point_residual":fixed_residual}
classifier_examples={
    "rotation about z-axis":(rotation_matrix([0,0,1],math.radians(40)),np.zeros(3)),
    "translation":(np.eye(3),np.array([.7,-.1,.2])),
    "central inversion":(-np.eye(3),np.zeros(3)),
    "plane reflection":reflection_affine([0,0,1],0),
    "twist":(R_twist,t_twist),
    "dilative rotation":(1.12*rotation_matrix([0,0,1],math.radians(25)),np.zeros(3)),
}
classifier_rows=[]
for name,(M,t) in classifier_examples.items(): classifier_rows.append({"example":name,**classify_affine(M,t)})
classifier_df=pd.DataFrame(classifier_rows)
assert classifier_df.loc[classifier_df["example"]=="twist","has_fixed_point"].item() is False
assert classifier_df.loc[classifier_df["example"]=="central inversion","orientation"].item()=="opposite"
assert classifier_df.loc[classifier_df["example"]=="dilative rotation","isometry"].item() is False
classifier_df


## Final Sanity Checks and Takeaways

The checks combine three layers: artifact integrity, numeric invariants, and symbolic identities. The notebook is complete only if all concept artifacts exist, have nonzero size, and agree with the intended geometry.

Takeaways:

- In space, signed volume replaces signed area as the sense detector.
- Central inversion is opposite in 3D because it is the product of three perpendicular plane reflections.
- Two reflections produce rotations or translations; three reflections produce the basic opposite motions.
- A twist is a direct isometry with no fixed point, visible as helical orbits around an axis.
- A dilative rotation packages rotation, dilation, central inversion, reflection, and rotatory inversion in one formula.
- Similarities and inversions in spheres explain why the final transformations preserve spheres.


In [ ]:
generated_artifacts=[PATHS[k] for k in ["orientation_html","orientation_checks","orientation_table","central_png","central_checks","reflection_html","reflection_checks","twist_html","twist_checks","twist_data","dilative_html","dilative_checks","dilative_data","special_cases","sphere_html","sphere_checks"]]
assert_artifacts(generated_artifacts,min_bytes=100)
loaded={name:json.loads(PATHS[key].read_text(encoding="utf-8")) for name,key in [("orientation","orientation_checks"),("central","central_checks"),("reflection","reflection_checks"),("twist","twist_checks"),("dilative","dilative_checks"),("sphere","sphere_checks")]}
assert loaded["orientation"]["max_distance_error"]<1e-12
assert loaded["central"]["central_inversion_determinant"]==-1
assert loaded["reflection"]["rotation_angle_error"]<1e-12
assert loaded["twist"]["fixed_point_least_squares_residual"]>.1
assert loaded["dilative"]["symbolic_determinant"]=="lambda**3"
assert loaded["sphere"]["unit_inversion_max_fit_residual"]<1e-12
storyboard_items=["direct/opposite spatial isometries via signed tetrahedron volume","central inversion as three perpendicular plane reflections","rotation and translation as products of two plane reflections","twist/screw traces with fixed-axis invariants","dilative rotation lab with special-case table","sphere-preservation checks for similarity and inversion in a sphere"]
visual_summary={"chapter":CHAPTER_NO,"title":"Isometry and Similarity in Euclidean Space","source_span":SOURCE_SPAN,"storyboard_items":storyboard_items,"artifact_count_before_summary":len(generated_artifacts),"key_invariants":{"orientation_max_distance_error":loaded["orientation"]["max_distance_error"],"central_inversion_determinant":loaded["central"]["central_inversion_determinant"],"reflection_rotation_angle_error":loaded["reflection"]["rotation_angle_error"],"twist_max_radius_drift":loaded["twist"]["max_radius_drift"],"dilative_symbolic_determinant":loaded["dilative"]["symbolic_determinant"],"sphere_inversion_max_fit_residual":loaded["sphere"]["unit_inversion_max_fit_residual"]}}
write_json(PATHS["visual_summary"],visual_summary)
manifest_core=generated_artifacts+[PATHS["visual_summary"]]
final_sanity={"chapter":CHAPTER_NO,"source_span":SOURCE_SPAN,"artifact_count":len(manifest_core)+2,"all_artifacts_nonzero":False,"checked_identities":["signed volume ratio equals det(M)","coordinate-plane reflection product equals central inversion","two reflection products double the mirror angle or gap","twist orbits keep radius and axial step invariants","det(lambda R) = lambda**3","inverted and similar images of a sphere fit spheres with tiny residual"],"max_numeric_residuals":{"orientation_distance":loaded["orientation"]["max_distance_error"],"reflection_angle":loaded["reflection"]["rotation_angle_error"],"twist_radius":loaded["twist"]["max_radius_drift"],"dilative_ratio":loaded["dilative"]["max_radial_ratio_error"],"sphere_fit":loaded["sphere"]["unit_inversion_max_fit_residual"]}}
write_json(PATHS["final_sanity"],final_sanity)
manifest_paths=manifest_core+[PATHS["final_sanity"]]
manifest_rows=[{"artifact":book_relative(path,BOOK_ROOT),"role":path.parent.name,"bytes":path.stat().st_size} for path in manifest_paths]
write_csv(PATHS["artifact_manifest"],manifest_rows)
final_paths=manifest_paths+[PATHS["artifact_manifest"]]
assert_artifacts(final_paths,min_bytes=100)
final_sanity["all_artifacts_nonzero"]=all(path.exists() and path.stat().st_size>100 for path in final_paths)
write_json(PATHS["final_sanity"],final_sanity)
manifest_rows=[{"artifact":book_relative(path,BOOK_ROOT),"role":path.parent.name,"bytes":path.stat().st_size} for path in manifest_paths]
write_csv(PATHS["artifact_manifest"],manifest_rows)
assert_artifacts(final_paths,min_bytes=100)
assert final_sanity["all_artifacts_nonzero"]
manifest_df=pd.DataFrame(manifest_rows)
display(manifest_df)
final_sanity
